## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [32]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

## First use pandas to read the stock market data

In [33]:
# Read the two CSV files into pandas DataFrames
nasdaq100_ca = pd.read_csv('nasdaq100_CA.csv')
price_change_df = pd.read_csv('nasdaq100_price_change.csv')

# Add the "ytd" column from price_change_df to ca_df by matching on 'symbol'
nasdaq100_ca = nasdaq100_ca.merge(price_change_df[['symbol', 'ytd']], on='symbol', how='left')

# Display the first few rows to verify
nasdaq100_ca.head()

,symbol,name,headQuarter,dateFirstAdded,cik,founded,ytd
0,AAPL,Apple Inc.,"Cupertino, CA",NaN,320193,1976-04-01,42.99992
1,ABNB,Airbnb,"San Francisco, CA",NaN,1559720,2008-08-01,68.66902
2,ADBE,Adobe Inc.,"San Jose, CA",NaN,796343,1982-12-01,57.22723
3,ADSK,Autodesk,"San Rafael, CA",NaN,769397,1982-01-30,10.02701
4,AMAT,Applied Materials,"Santa Clara, CA",NaN,6951,1967-11-10,55.46366


## Use OpenAI to classify each company

In [34]:
# Define the list of sectors for classification
sectors = [
    "Technology", "Consumer Cyclical", "Industrials", "Utilities", "Healthcare",
    "Communication", "Energy", "Consumer Defensive", "Real Estate", "Financial"
]

# Prepare prompts for each company
def build_prompt(row):
    return (
        f"Classify the following company into one of these sectors: {', '.join(sectors)}.\n"
        f"Company Name: {row['name']}\n"
        f"Headquarters: {row['headQuarter']}\n"
        f"Description: (If you know it, otherwise just use the name and headquarters)\n"
        f"Respond with only the sector name."
    )

prompts = nasdaq100_ca.apply(build_prompt, axis=1).tolist()

# Call OpenAI API to classify each company
sector_results = []
for prompt in prompts:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    # Extract the sector from the response
    sector = response.choices[0].message.content.strip()
    sector_results.append(sector)

# Add the sector classification to the DataFrame
nasdaq100_ca['sector'] = sector_results

# Display the first few rows to verify
nasdaq100_ca.head()

,symbol,name,headQuarter,dateFirstAdded,cik,founded,ytd,sector
0,AAPL,Apple Inc.,"Cupertino, CA",NaN,320193,1976-04-01,42.99992,Technology
1,ABNB,Airbnb,"San Francisco, CA",NaN,1559720,2008-08-01,68.66902,Consumer Cyclical
2,ADBE,Adobe Inc.,"San Jose, CA",NaN,796343,1982-12-01,57.22723,Technology
3,ADSK,Autodesk,"San Rafael, CA",NaN,769397,1982-01-30,10.02701,Technology
4,AMAT,Applied Materials,"Santa Clara, CA",NaN,6951,1967-11-10,55.46366,Technology


## Use AI to recommend best sectors and stocks YTD

In [35]:
# Prompt to get stock recommendations
prompt = f'''Provide summary information about Nasdaq-100 stock performance year to date (YTD) of companies headquartered in CA, recommending the two best sectors and two or more companies per sector.
            Company data: {nasdaq100_ca} 
'''

# Get the model response
response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{ "role": "user", "content": prompt}],
        temperature=0.0,
    )

# Store the output as a variable and print the recommendations
stock_recommendations = response.choices[0].message.content
print(stock_recommendations)

The Nasdaq-100 stock performance year to date (YTD) for companies headquartered in California shows that the Technology sector has been performing well, with companies like Nvidia (NVDA) and Meta Platforms (META) showing significant gains. The Consumer Cyclical sector also has strong performers such as Airbnb (ABNB) and Lucid Motors (LCID).

The two best sectors based on YTD performance are:
1. Technology:
   - Nvidia (NVDA) with a YTD performance of 217.27%
   - Meta Platforms (META) with a YTD performance of 153.78%

2. Consumer Cyclical:
   - Airbnb (ABNB) with a YTD performance of 68.67%
   - Lucid Motors (LCID) with a YTD performance of 3.89%
